# LULC Change Detection – Tirumala and Tirupati (TTD Region), 2000–2025  (final version)
**Python + Google Earth Engine, Landsat 5/7/8/9, 5-year steps. No manual point collection.**

Run from the top. Before running, put your own Earth Engine project id in the first code cell. This copy is anonymised for review.
The cells that call Earth Engine can be re-run after an interruption (they keep what is already done).

**What this version does**
| # | Step | Detail |
|---|---|---|
| 1 | Study areas | Two rectangles that do not overlap (Tirupati = 79.39–79.60 E) |
| 2 | Composites | Same rule for every year: Jan–Apr, three years (2000: 2000–2002; others: year−1, year, year+1); Landsat 5/7 harmonised to Landsat 8 (Roy et al. 2016) |
| 3 | Labels | Automatic, from GHSL (built-up), JRC Global Surface Water (water), Hansen Global Forest Change v1.13 (forest) |
| 4 | Classifier | ONE Random Forest trained on points pooled from all years and both areas, 3x3 majority filter, spatial-block validation split |
| 5 | Accuracy | OA, Kappa, UA, PA of the Random Forest maps at held-out points |
| 6 | Final maps | Built-up and water from the Random Forest. **Forest cover = Hansen canopy ≥ 20% in 2000 minus loss by that year.** Everything else = other land |
| 7 | Change | Area per class, net change, transition matrices |

| Notebook section | Flowchart step |
|---|---|
| 0. Setup | – |
| 1. Study areas | Step 1 |
| 2. Landsat data and pre-processing | Steps 2 and 3 |
| 3. Automatic training labels | Step 4 |
| 4. Pooled Random Forest | Step 5 |
| 5. Accuracy assessment | Step 6 |
| 5b. Final maps (canopy-based forest layer) | Step 6 |
| 6. Area statistics (+ GHSL and JRC comparison) | Step 7 |
| 7. Change detection | Step 7 |
| 8. Save results (run this BEFORE the figures) | Step 8 |
| 9. Figures (saved and zipped) | Step 8 |

## 0. Setup

In [ ]:
!pip install -q geemap

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Needed in Colab so that widgets/maps work
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import geemap.coreutils as cu
cu.get_google_maps_api_key = lambda *a, **k: None     # skip the secret lookup (Colab fix)

ee.Authenticate()
ee.Initialize(project='YOUR_EARTH_ENGINE_PROJECT_ID')      # put your own Earth Engine project id here

## 1. Study areas (Step 1)
Two rectangular study areas, analysed separately. Coordinates are `[West, South, East, North]` in degrees.
They now **share only an edge (79.39 E)** and do not overlap. The Tirupati box extends to 79.60 E to include Renigunta.

In [ ]:
BOX = {
    'Tirumala': [79.29, 13.64, 79.39, 13.74],
    'Tirupati': [79.39, 13.55, 79.60, 13.69],
}
AREAS = {name: ee.Geometry.Rectangle(b) for name, b in BOX.items()}
YEARS = [2000, 2005, 2010, 2015, 2020, 2025]

# Classes (ids fixed: 1 built-up, 3 water, 4 forest, 5 other land). Roads are NOT mapped (see Section 3).
CLASSES = {1: 'Built-up', 3: 'Water bodies', 4: 'Forest cover', 5: 'Other land'}
CLASS_IDS = list(CLASSES.keys())
PALETTE = ['d62728', '000000', '1f77b4', '2ca02c', 'e0d6a8']   # index = class id - 1 (id 2 unused)

SEED = 42
OA_MIN, KAPPA_MIN = 0.85, 0.75      # acceptance thresholds (flowchart decision box). Kappa lowered from 0.80 to 0.75 AFTER the first v2 run: say so in Methods

# overlap check (client side, no Earth Engine call)
(w1, s1, e1, n1), (w2, s2, e2, n2) = BOX['Tirumala'], BOX['Tirupati']
ow, oh = min(e1, e2) - max(w1, w2), min(n1, n2) - max(s1, s2)
print('Overlap area:', 'NONE' if (ow <= 0 or oh <= 0) else f'{ow:.3f} x {oh:.3f} deg  <-- FIX THE BOXES')
for name, geom in AREAS.items():
    print(f'{name}: {geom.area().divide(1e6).getInfo():.1f} km2')

## 2. Landsat data and pre-processing (Steps 2 and 3)
* Landsat Collection 2 Level 2 **surface reflectance**, cloud and shadow masked with `QA_PIXEL`.
* **Same rule for every year:** January–April scenes only, from **three years** (2000: 2000–2002; other years: year−1, year, year+1). The window is never widened, so all maps use the same season.
* Sensors: Landsat 5+7 (2000, 2005, 2010), Landsat 8 (2015, 2020), Landsat 8+9 (2025).
* **Harmonisation:** Landsat 5/7 reflectance is converted to Landsat 8 level with the ETM+→OLI regression of Roy et al. (2016). The same coefficients are applied to Landsat 5 (assumed similar to Landsat 7). Switch off with `HARMONISE = False` to test its effect.
* One **median composite** per area and year. Bands used: 6 reflectance bands + NDVI, NDBI, MNDWI.

In [ ]:
SENSORS = {
    2000: ['LT05', 'LE07'], 2005: ['LT05', 'LE07'], 2010: ['LT05', 'LE07'],
    2015: ['LC08'], 2020: ['LC08'], 2025: ['LC08', 'LC09'],
}
COLLECTION = {
    'LT05': 'LANDSAT/LT05/C02/T1_L2', 'LE07': 'LANDSAT/LE07/C02/T1_L2',
    'LC08': 'LANDSAT/LC08/C02/T1_L2', 'LC09': 'LANDSAT/LC09/C02/T1_L2',
}
BANDS_IN = {   # blue, green, red, nir, swir1, swir2
    'LT05': ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
    'LE07': ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
    'LC08': ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
    'LC09': ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
}
BANDS_OUT = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
BANDS = BANDS_OUT + ['NDVI', 'NDBI', 'MNDWI']          # bands used for classification

MONTHS = (1, 4)          # January to April, every year
MIN_SCENES = 3           # only a warning now; the window is NOT widened
HARMONISE = True

# Roy et al. (2016, Remote Sensing of Environment 185:57-70), Table 2, ETM+ -> OLI, OLS:  OLI = intercept + slope * ETM+
RT_SLOPE = [0.8474, 0.8483, 0.9047, 0.8462, 0.8937, 0.9071]      # blue, green, red, nir, swir1, swir2
RT_ITCP = [0.0003, 0.0088, 0.0061, 0.0412, 0.0254, 0.0172]


def year_range(year):
    return (2000, 2002) if year == 2000 else (year - 1, year + 1)


def prep(img, sensor):
    """Mask cloud/shadow, scale to surface reflectance, rename bands, harmonise Landsat 5/7 to Landsat 8."""
    qa = img.select('QA_PIXEL')
    clear = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))   # bit 3 = cloud, bit 4 = shadow
    sr = img.select(BANDS_IN[sensor]).multiply(0.0000275).add(-0.2).rename(BANDS_OUT)
    if HARMONISE and sensor in ('LT05', 'LE07'):
        sr = sr.multiply(ee.Image.constant(RT_SLOPE)).add(ee.Image.constant(RT_ITCP)).rename(BANDS_OUT)
    return sr.updateMask(clear)


def make_prep(sensor):
    return lambda im: prep(im, sensor)


def add_indices(img):
    ndvi = img.normalizedDifference(['nir', 'red']).rename('NDVI')
    ndbi = img.normalizedDifference(['swir1', 'nir']).rename('NDBI')
    mndwi = img.normalizedDifference(['green', 'swir1']).rename('MNDWI')
    return img.addBands([ndvi, ndbi, mndwi])


def get_composite(year, region):
    y0, y1 = year_range(year)
    col = None
    for sensor in SENSORS[year]:
        c = (ee.ImageCollection(COLLECTION[sensor])
             .filterBounds(region)
             .filterDate(f'{y0}-01-01', f'{y1}-12-31')
             .filter(ee.Filter.calendarRange(MONTHS[0], MONTHS[1], 'month'))
             .map(make_prep(sensor)))
        col = c if col is None else col.merge(c)
    n = col.size().getInfo()
    if n < MIN_SCENES:
        print(f'  WARNING: only {n} scenes for {year}. Mention this in Limitations.')
    img = add_indices(col.median().clip(region)).toFloat()
    return img, n


COMPOSITES, rows = {}, []
for area, geom in AREAS.items():
    for year in YEARS:
        img, n = get_composite(year, geom)
        COMPOSITES[(area, year)] = img
        y0, y1 = year_range(year)
        rows.append({'Area': area, 'Year': year, 'Sensors': '+'.join(SENSORS[year]),
                     'Image years': f'{y0}-{y1}', 'Months': 'Jan-Apr', 'Scenes used': n})
scene_df = pd.DataFrame(rows)
scene_df

**Check the table above.** All rows must say Jan–Apr. A row with a very low scene count (under about 6) means a noisy composite; if 2000 is still poor, use the fallback (2005–2025 only) and say so in the chapter.

## 3. Automatic training labels (Step 4 and input to Step 5)
Labels come from three free reference datasets, one per class. Only *confident* pixels are used.

| Class | Reference dataset | Rule for a confident label in year *Y* |
|---|---|---|
| Built-up | GHSL GHS-BUILT-S R2023A (100 m, epochs every 5 years) | at least 40% of the 100 m cell is built-up |
| Water bodies | JRC Global Surface Water, yearly history (30 m, to 2021) | "permanent" water in year *Y* (2021 used for 2025) |
| Forest cover | Hansen Global Forest Change **v1.13 (2025)** | tree cover ≥ 60% in 2000 and not lost by year *Y*; no built-up; not water |
| Other land | the three datasets above | no built-up, no water, tree cover < 20% |

**Know these limits (write them in the chapter)**
* Accuracy = agreement with these datasets, not with field data.
* Hansen shows tree-cover **loss** only (no gain), so the forest label is almost the same in every year.
* Pixels with 20–60% tree cover get no label but are still classified, so forest/other changes there are the least certain.
* The GHSL catalog describes its epochs as interpolated or extrapolated. The 2025 epoch is, as far as I know, a projection, not an observation. **Verify in Pesaresi & Politis (2023) before you write the Methods.** To test it, set `GHSL_EPOCH[2025] = 2020`.
* **Roads are not mapped** (no free historical road data; roads are narrower than 30 m).

In [ ]:
LABEL_CLASSES = [1, 3, 4, 5]
GHSL_BUILT_MIN = 4000               # m2 of built surface per 100 m cell (40%)
TREE_MIN, TREE_ANY = 60, 20         # % tree cover: confident forest / "possibly tree-covered"
POINTS_PER_CLASS = 250

HANSEN_ID = 'UMD/hansen/global_forest_change_2025_v1_13'
hansen = ee.Image(HANSEN_ID)
GHSL_EPOCH = {y: y for y in YEARS}  # set GHSL_EPOCH[2025] = 2020 to avoid the projected 2025 epoch


def reference_labels(year):
    """Confident land cover labels for one year from GHSL, JRC Global Surface Water and Hansen."""
    built_surf = ee.Image(f'JRC/GHSL/P2023A/GHS_BUILT_S/{GHSL_EPOCH[year]}').select('built_surface').unmask(0)

    wy = min(year, 2021)            # yearly water history ends in 2021
    gsw = (ee.ImageCollection('JRC/GSW1_4/YearlyHistory')
           .filterDate(f'{wy}-01-01', f'{wy}-12-31').first().select('waterClass').unmask(0))

    tree = hansen.select('treecover2000').unmask(0)
    loss = hansen.select('lossyear').unmask(0)
    lost = loss.gt(0).And(loss.add(2000).lte(year))      # forest lost on or before this year

    water = gsw.eq(3)                                     # 3 = permanent water only
    nobuilt = built_surf.lt(100)
    built = built_surf.gte(GHSL_BUILT_MIN).And(water.Not())
    forest = tree.gte(TREE_MIN).And(lost.Not()).And(nobuilt).And(water.Not())
    treeish = tree.gte(TREE_ANY).And(lost.Not())
    other = nobuilt.And(water.Not()).And(treeish.Not())

    lab = ee.Image(0).where(other, 5).where(forest, 4).where(built, 1).where(water, 3)
    return lab.updateMask(lab.gt(0)).rename('class').toByte()


_POINT_CACHE = {}


def get_points(area, year):
    """Random confident points per class for one area and year (cached)."""
    key = (area, year)
    if key not in _POINT_CACHE:
        _POINT_CACHE[key] = reference_labels(year).stratifiedSample(
            numPoints=0, classBand='class', region=AREAS[area], scale=30,
            classValues=LABEL_CLASSES, classPoints=[POINTS_PER_CLASS] * len(LABEL_CLASSES),
            seed=SEED, geometries=True, tileScale=4)
    return _POINT_CACHE[key]

## 4. One Random Forest for all years (Step 5)
**Why:** in version 1 every year had its own classifier, so the maps differed partly because the classifiers differed. Now:
1. Samples from **all years and both areas** are pooled (Tirumala alone has only about 20–50 water/built-up labels per year).
2. **One** Random Forest (200 trees, 9 bands) is trained once and applied to all 12 composites.
3. A **3x3 majority filter** removes salt-and-pepper pixels (same filter for every year).
4. Train/validation split is by **spatial blocks (about 1 km)**, not random points, so validation points are not right next to training points.

Switches: `POOL_AREAS = False` trains one model per area; `MODE_FILTER = False` keeps the raw pixel map.

In [ ]:
POOL_AREAS = True
MODE_FILTER = True
BLOCK_DEG = 0.01          # about 1 km blocks for the spatial split
VALID_FRAC = 0.30
MIN_TRAIN, MIN_VALID = 40, 10     # per class, used as a sanity check on the split


# ---------- helper functions (no Earth Engine calls) ----------
def add_block(df):
    df = df.copy()
    df['block'] = (np.floor(df['lon'] / BLOCK_DEG).astype(int).astype(str) + '_' +
                   np.floor(df['lat'] / BLOCK_DEG).astype(int).astype(str))
    return df


def spatial_split(df, tries=500):
    """Whole blocks go to validation. Each (area, class) pair - years pooled - should get about VALID_FRAC of its points
    in validation, and no pair may end up with none."""
    blocks = np.array(sorted(df['block'].unique()))
    key = df['Area'] + '|' + df['ref'].astype(str)
    best = None
    for s in range(SEED, SEED + tries):
        rng = np.random.RandomState(s)
        vb = set(blocks[rng.rand(len(blocks)) < VALID_FRAC])
        is_valid = df['block'].isin(vb)
        frac = is_valid.groupby(key).mean()
        score = float((frac - VALID_FRAC).abs().sum()) + 10.0 * int(((frac == 0) | (frac == 1)).sum())
        if best is None or score < best[0]:
            best = (score, s, is_valid)
    return best


def metrics(y, p, classes):
    """OA, Kappa, UA, PA from reference labels y and mapped labels p."""
    idx = {c: i for i, c in enumerate(classes)}
    cm = np.zeros((len(classes), len(classes)))
    for a, b in zip(y, p):
        if a in idx and b in idx:
            cm[idx[a], idx[b]] += 1
    n = cm.sum()
    if n == 0:
        return None
    oa = np.trace(cm) / n
    pe = (cm.sum(0) * cm.sum(1)).sum() / n ** 2
    kappa = (oa - pe) / (1 - pe) if pe < 1 else np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        ua = np.diag(cm) / cm.sum(0)       # user's accuracy (columns = mapped)
        pa = np.diag(cm) / cm.sum(1)       # producer's accuracy (rows = reference)
    return {'OA': oa, 'Kappa': kappa, 'UA': ua, 'PA': pa, 'n': int(n), 'n_ref': cm.sum(1)}

In [ ]:
# ---------- sample the 12 composites at the label points (resumable: re-run this cell after an interruption) ----------
SAMPLE_ROWS = globals().get('SAMPLE_ROWS', {})


def get_samples(area, year):
    img = COMPOSITES[(area, year)].select(BANDS)
    fc = img.sampleRegions(collection=get_points(area, year), properties=['class'],
                           scale=30, tileScale=4, geometries=True)
    out = []
    for f in fc.getInfo()['features']:
        p = f['properties']
        if any(p.get(b) is None for b in BANDS):
            continue
        lon, lat = f['geometry']['coordinates'][:2]
        out.append(dict(Area=area, Year=year, ref=int(p['class']), lon=lon, lat=lat,
                        **{b: round(float(p[b]), 5) for b in BANDS}))
    return out


for area in AREAS:
    for year in YEARS:
        if (area, year) in SAMPLE_ROWS:
            continue
        print(f'Sampling {area} {year} ...')
        SAMPLE_ROWS[(area, year)] = get_samples(area, year)

samples = add_block(pd.DataFrame([r for rows in SAMPLE_ROWS.values() for r in rows]))
sample_counts = (samples.pivot_table(index=['Area', 'Year'], columns='ref', values='lon', aggfunc='count')
                 .rename(columns=CLASSES).fillna(0).astype(int))
sample_counts

In [ ]:
# Diagnostic (no Earth Engine call): mean NDVI of the LABELLED points per area, year and class.
# If "Forest cover" and "Other land" NDVI move up and down together from year to year, the composites differ
# in greenness (rainfall/season), which makes the forest/other boundary move even with one classifier.
ndvi_by_class = samples.groupby(['Area', 'Year', 'ref'])['NDVI'].mean().unstack('ref').rename(columns=CLASSES).round(3)
ndvi_by_class

In [ ]:
# ---------- train ONE Random Forest per group (default: all years, both areas) and classify all 12 maps ----------
GROUPS = [list(AREAS)] if POOL_AREAS else [[a] for a in AREAS]
CLASSIFIED_RF, VALID_ROWS, SPLIT_INFO = {}, {}, []

for group in GROUPS:
    sub = samples[samples['Area'].isin(group)]
    score, seed_used, is_valid = spatial_split(sub)
    train, valid = sub[~is_valid], sub[is_valid]
    ntr, nva = train['ref'].value_counts(), valid['ref'].value_counts()
    for c in LABEL_CLASSES:
        if ntr.get(c, 0) < MIN_TRAIN or nva.get(c, 0) < MIN_VALID:
            print(f'  WARNING {group}: class {CLASSES[c]} has {ntr.get(c, 0)} train / {nva.get(c, 0)} validation points')
    print('Group', group, '- validation points per area/class (all years):')
    print(valid.groupby(['Area', 'ref']).size().unstack(fill_value=0).rename(columns=CLASSES))
    SPLIT_INFO.append({'Group': '+'.join(group), 'Split seed': seed_used, 'Train points': len(train),
                       'Validation points': len(valid),
                       **{f'Train {CLASSES[c]}': int(ntr.get(c, 0)) for c in LABEL_CLASSES}})

    train_fc = ee.FeatureCollection([
        ee.Feature(None, {**{b: float(r[b]) for b in BANDS}, 'class': int(r['ref'])})
        for r in train.to_dict('records')])
    rf = ee.Classifier.smileRandomForest(numberOfTrees=200, seed=SEED).train(
        features=train_fc, classProperty='class', inputProperties=BANDS)

    for area in group:
        VALID_ROWS[area] = valid[valid['Area'] == area]
        for year in YEARS:
            raw = COMPOSITES[(area, year)].select(BANDS).classify(rf).rename('lulc').toByte()
            # 3x3 majority filter (45 m square radius = 3x3 pixels at 30 m); same for every year
            final = raw.focalMode(radius=45, kernelType='square', units='meters') if MODE_FILTER else raw
            CLASSIFIED_RF[(area, year)] = final.rename('lulc').toByte()

pd.DataFrame(SPLIT_INFO)

## 5. Accuracy assessment (Step 6)
OA, Kappa, UA, PA are computed on the **Random Forest maps** (after the 3x3 filter) at the held-out validation points, one row per map.
`n ...` columns show how many validation points each class has: UA/PA based on fewer than about 20 points are not reliable.
These numbers measure **agreement with the reference datasets**, not field truth.
Acceptance rule: OA ≥ 0.85 and Kappa ≥ 0.75. The Kappa limit was set to 0.75 after the first results were seen (it was 0.80 in the first plan): say this in the chapter.
Built-up and water stay unchanged in the final maps (section 5b), so their UA and PA carry over. The forest layer of the final maps is not assessed (its reference labels come from the same Hansen data, so the test would be circular).

In [ ]:
ACC = globals().get('ACC', {})
PRED = globals().get('PRED', {})

for area in AREAS:
    for year in YEARS:
        if (area, year) in ACC:
            continue
        v = VALID_ROWS[area][VALID_ROWS[area]['Year'] == year]
        fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {'ref': int(r.ref)})
                                   for r in v.itertuples()])
        out = CLASSIFIED_RF[(area, year)].sampleRegions(collection=fc, properties=['ref'], scale=30, tileScale=4).getInfo()['features']
        y = [int(f['properties']['ref']) for f in out if f['properties'].get('lulc') is not None]
        p = [int(f['properties']['lulc']) for f in out if f['properties'].get('lulc') is not None]
        PRED[(area, year)] = (y, p)
        m = metrics(y, p, CLASS_IDS)
        row = {'Area': area, 'Year': year, 'Validation points': m['n'], 'OA': round(m['OA'], 3), 'Kappa': round(m['Kappa'], 3)}
        for i, c in enumerate(CLASS_IDS):
            row[f'UA {CLASSES[c]}'] = round(float(m['UA'][i]), 3)
            row[f'PA {CLASSES[c]}'] = round(float(m['PA'][i]), 3)
            row[f'n {CLASSES[c]}'] = int(m['n_ref'][i])
        row['Check'] = 'OK' if (m['OA'] >= OA_MIN and m['Kappa'] >= KAPPA_MIN) else 'BELOW THRESHOLD'
        ACC[(area, year)] = row
        print(f'{area} {year}: OA {row["OA"]}  Kappa {row["Kappa"]}')

acc_df = pd.DataFrame(list(ACC.values()))

# Accuracy pooled over the six years, per area: more points per class than any single map
pool_rows = []
for area in AREAS:
    y = sum((PRED[(area, yr)][0] for yr in YEARS), [])
    p = sum((PRED[(area, yr)][1] for yr in YEARS), [])
    m = metrics(y, p, CLASS_IDS)
    row = {'Area': area, 'Points (6 years)': m['n'], 'OA': round(m['OA'], 3), 'Kappa': round(m['Kappa'], 3)}
    for i, c in enumerate(CLASS_IDS):
        row[f'UA {CLASSES[c]}'] = round(float(m['UA'][i]), 3)
        row[f'PA {CLASSES[c]}'] = round(float(m['PA'][i]), 3)
        row[f'n {CLASSES[c]}'] = int(m['n_ref'][i])
    pool_rows.append(row)
pooled_df = pd.DataFrame(pool_rows)
display(acc_df)
pooled_df

**Chapter tables 5 and 6.** Per-map OA and Kappa (Table 5) and pooled UA and PA per class (Table 6), also saved as `ttd_accuracy_tables.xlsx`. Reference labels: GHSL (built-up), JRC (water), Hansen (forest); the figures are agreement with these datasets.

In [ ]:
# Table A: per-map OA and Kappa (Random Forest maps, 3x3 filter applied), with the acceptance check.
# Table B: pooled (six years) UA and PA per class and area.
# Reference labels: built-up = GHSL, water = JRC, forest = Hansen canopy (>=60 %), other land = remainder.
tableA = acc_df[['Area', 'Year', 'Validation points', 'OA', 'Kappa', 'Check']].copy()
tableA['OA'] = tableA['OA'].map('{:.3f}'.format)
tableA['Kappa'] = tableA['Kappa'].map('{:.3f}'.format)

names = list(CLASSES.values())
if 'pooled_df' in globals():
    rowsB = []
    for _, r in pooled_df.iterrows():
        for n in names:
            rowsB.append({'Area': r['Area'], 'Class': n, 'Reference points': int(r[f'n {n}']),
                          'UA': round(float(r[f'UA {n}']), 3), 'PA': round(float(r[f'PA {n}']), 3)})
    tableB = pd.DataFrame(rowsB)
else:
    tableB = None

print('Table A - per-map accuracy')
display(tableA)
print('Maps below the OA or Kappa threshold:', (acc_df['Check'] != 'OK').sum(), 'of', len(acc_df))
if tableB is not None:
    print('\nTable B - pooled UA and PA')
    display(tableB)

with pd.ExcelWriter('ttd_accuracy_tables.xlsx') as xw:
    tableA.to_excel(xw, sheet_name='TableA_OA_Kappa', index=False)
    if tableB is not None:
        tableB.to_excel(xw, sheet_name='TableB_UA_PA', index=False)
print('Saved ttd_accuracy_tables.xlsx')


## 5b. Final maps: forest cover from Hansen canopy
In the Random Forest maps the forest area jumped by several km² between years because the forest labels come from one static canopy map, so the classifier cannot learn real vegetation change.
The final maps therefore take **built-up and water from the Random Forest** and **forest cover from Hansen tree canopy**:
forest = canopy ≥ 20% in 2000 and not lost on or before the mapped year; it never overrides built-up or water. All other pixels are other land.
Consequences: forest change = Hansen tree-cover loss (plus pixels taken by built-up or water); forest gain is not measured; the 20% threshold is a choice.
The Random Forest maps stay in `CLASSIFIED_RF`; the final maps are in `CLASSIFIED` and are used from here on.

In [ ]:
tree2000 = hansen.select('treecover2000').unmask(0)
loss = hansen.select('lossyear').unmask(0)

CLASSIFIED = {}
for (area, year), rf_map in CLASSIFIED_RF.items():
    lost = loss.gt(0).And(loss.add(2000).lte(year))            # tree cover lost on or before this year
    tree = tree2000.gte(TREE_ANY).And(lost.Not())              # TREE_ANY = 20 (% canopy)
    final = (ee.Image(5).where(tree, 4)                        # 5 other land, 4 forest cover
             .where(rf_map.eq(3), 3)                           # water from the Random Forest
             .where(rf_map.eq(1), 1)                           # built-up from the Random Forest
             .updateMask(rf_map.mask()).rename('lulc').toByte())
    CLASSIFIED[(area, year)] = final

# area and transition results must be recomputed for the final maps
for d in ('AREA_CACHE', 'TRANSITIONS'):
    globals().get(d, {}).clear()
print('Final maps built:', len(CLASSIFIED))

## 6. Area statistics (Step 7: area change)

In [ ]:
def class_areas(classified, region):
    """Area in km2 of each class inside the region."""
    stats = (ee.Image.pixelArea().divide(1e6).addBands(classified)
             .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
                           geometry=region, scale=30, maxPixels=1e10))
    groups = ee.List(stats.get('groups')).getInfo()
    return {int(g['class']): g['sum'] for g in groups}


AREA_CACHE = globals().get('AREA_CACHE', {})
for area in AREAS:
    for year in YEARS:
        if (area, year) not in AREA_CACHE:
            print(f'Area statistics {area} {year} ...')
            AREA_CACHE[(area, year)] = class_areas(CLASSIFIED[(area, year)], AREAS[area])

area_df = pd.DataFrame([{'Area': a, 'Year': y, 'Class': CLASSES[c], 'Area_km2': AREA_CACHE[(a, y)].get(c, 0.0)}
                        for (a, y) in AREA_CACHE for c in CLASS_IDS])
four = [CLASSES[c] for c in CLASS_IDS if c != 5]          # classes of interest (leave out "Other land")

area_tables = {}
for area in AREAS:
    t = (area_df[area_df['Area'] == area]
         .pivot(index='Class', columns='Year', values='Area_km2')
         .reindex(index=list(CLASSES.values()), columns=YEARS).round(2))
    t['Change 2000-2025 (km2)'] = (t[2025] - t[2000]).round(2)
    t['Change 2000-2025 (%)'] = ((t[2025] - t[2000]) / t[2000] * 100).replace([np.inf, -np.inf], np.nan).round(1)
    area_tables[area] = t
    print(f'\n{area} - area by class (km2)')
    display(t)

**Independent comparison (GHSL and JRC).** Built-up: area of 100 m GHSL cells with at least 40% (the label rule) and at least 10% built surface, next to the mapped built-up area.
Water: JRC permanent and seasonal water, next to the mapped water (2021 is used for 2025). All values in km². Use this table in the Results and Discussion; it is a comparison, not an accuracy test.

In [ ]:
rows = []
for area in AREAS:
    for year in YEARS:
        g = ee.Image(f'JRC/GHSL/P2023A/GHS_BUILT_S/{GHSL_EPOCH[year]}').select('built_surface').unmask(0)
        cells_ = ee.Image.cat([g.gte(4000).rename('c40'), g.gte(1000).rename('c10')]).multiply(0.01)   # one 100 m cell = 0.01 km2
        s = cells_.reduceRegion(ee.Reducer.sum(), geometry=AREAS[area], scale=100, maxPixels=1e10).getInfo()
        wy = min(year, 2021)
        gsw = (ee.ImageCollection('JRC/GSW1_4/YearlyHistory')
               .filterDate(f'{wy}-01-01', f'{wy}-12-31').first().select('waterClass'))
        w = (ee.Image.cat([gsw.eq(3).rename('perm'), gsw.eq(2).rename('seas')])
             .multiply(ee.Image.pixelArea().divide(1e6))
             .reduceRegion(ee.Reducer.sum(), geometry=AREAS[area], scale=30, maxPixels=1e10).getInfo())
        rows.append({'Area': area, 'Year': year,
                     'GHSL cells >=40% built': round(s['c40'], 2), 'GHSL cells >=10% built': round(s['c10'], 2),
                     'Mapped built-up': round(float(area_tables[area].loc['Built-up', year]), 2),
                     'JRC permanent water': round(w['perm'], 2), 'JRC seasonal water': round(w['seas'], 2),
                     'Mapped water': round(float(area_tables[area].loc['Water bodies', year]), 2)})
ghsl_jrc = pd.DataFrame(rows)
ghsl_jrc

## 7. Change detection (Step 7: class transitions)
Transition matrices between consecutive dates and overall (2000→2025). **Rows = earlier year, columns = later year**, km². Large off-diagonal numbers in both directions (A→B and B→A) mean noise, not real change.

In [ ]:
def transition_matrix(area, y1, y2):
    a, b = CLASSIFIED[(area, y1)], CLASSIFIED[(area, y2)]
    change = a.multiply(10).add(b).rename('chg')          # e.g. 3 -> 1 becomes 31
    stats = (ee.Image.pixelArea().divide(1e6).addBands(change)
             .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName='chg'),
                           geometry=AREAS[area], scale=30, maxPixels=1e10))
    names = list(CLASSES.values())
    m = pd.DataFrame(0.0, index=names, columns=names)
    for g in ee.List(stats.get('groups')).getInfo():
        code_ = int(g['chg'])
        m.loc[CLASSES[code_ // 10], CLASSES[code_ % 10]] = g['sum']
    return m.round(2)


PAIRS = [(YEARS[i], YEARS[i + 1]) for i in range(len(YEARS) - 1)] + [(2000, 2025)]
TRANSITIONS = globals().get('TRANSITIONS', {})
for area in AREAS:
    for y1, y2 in PAIRS:
        if (area, y1, y2) not in TRANSITIONS:
            print(f'Transition {area} {y1}->{y2} ...')
            TRANSITIONS[(area, y1, y2)] = transition_matrix(area, y1, y2)

for area in AREAS:
    print(f'\n{area}: transition matrix 2000 -> 2025 (km2)')
    display(TRANSITIONS[(area, 2000, 2025)])

## 7b. Forest checks (run before Section 8)
1. **Independent check.** The final forest layer for 2020 and 2025 is compared with ESA WorldCover tree cover (300 tree + 300 non-tree points per area and year, seed 42; the 2025 map is compared with WorldCover 2021). WorldCover defines tree cover differently from the 20% Hansen rule, so this is agreement between two definitions, not error against ground truth.
2. **Diagnostic.** For WorldCover tree points: what our map says, and what the 2000 Hansen canopy was.
3. **Why the Hansen forest layer is used.** Forest area per epoch in the final maps and in the Random Forest maps.

In [ ]:
# Independent accuracy check of the FINAL forest layer against ESA WorldCover (10 m, tree cover class = 10).
# WorldCover exists only for 2020 (v100) and 2021 (v200), so this checks the 2020 and 2025 final maps.
# Points are drawn automatically (no manual points), seed 42. Forest = class 4 in the final map; everything else = not forest.
WC = {2020: 'ESA/WorldCover/v100/2020', 2025: 'ESA/WorldCover/v200/2021'}   # 2025 map is compared with the 2021 WorldCover
N_PER_CLASS = 300

FOREST_CHECK_PRED = globals().get('FOREST_CHECK_PRED', {})
fc_rows = []
for area in AREAS:
    for year, wc_id in WC.items():
        wc_tree = ee.Image(wc_id).select('Map').eq(10).rename('ref').toByte()          # 1 = tree cover, 0 = other
        mapped = CLASSIFIED[(area, year)].eq(4).rename('map').toByte()                 # 1 = forest in our final map
        pts = (wc_tree.addBands(mapped)
               .stratifiedSample(numPoints=0, classBand='ref', region=AREAS[area], scale=30,
                                 classValues=[0, 1], classPoints=[N_PER_CLASS, N_PER_CLASS],
                                 seed=SEED, tileScale=4, geometries=False))
        feats = pts.getInfo()['features']
        y = [int(f['properties']['ref']) for f in feats if f['properties'].get('map') is not None]
        p = [int(f['properties']['map']) for f in feats if f['properties'].get('map') is not None]
        FOREST_CHECK_PRED[(area, year)] = (y, p)
        m = metrics(y, p, [0, 1])
        fc_rows.append({'Area': area, 'Map year': year, 'WorldCover year': 2021 if year == 2025 else 2020,
                        'Points': m['n'], 'OA': round(m['OA'], 3), 'Kappa': round(m['Kappa'], 3),
                        'UA Forest': round(float(m['UA'][1]), 3), 'PA Forest': round(float(m['PA'][1]), 3)})
forest_check = pd.DataFrame(fc_rows)
forest_check
# Notes for the chapter:
#  * Sampling is stratified (equal tree / non-tree points), so UA/PA are per-class rates, not area-weighted estimates.
#  * WorldCover "tree cover" has its own definition (canopy > 10 % and > 5 m) and is not the same as the 20 % Hansen rule.
#  * Only 2020 and 2025 can be checked; 2000-2015 forest has no independent 10 m reference.
#  * Add forest_check to the Excel writer in Section 8:  forest_check.to_excel(xw, sheet_name='Forest_check', index=False)


In [ ]:
# For WorldCover tree-cover points, what does OUR final map say, and what was the 2000 Hansen canopy there?
rows = []
for area in AREAS:
    for year, wc_id in WC.items():
        wc_tree = ee.Image(wc_id).select('Map').eq(10).rename('ref').toByte()
        stack = (wc_tree.addBands(CLASSIFIED[(area, year)].rename('our'))
                 .addBands(CLASSIFIED_RF[(area, year)].rename('rf'))
                 .addBands(hansen.select('treecover2000').unmask(0).rename('canopy2000')))
        pts = stack.stratifiedSample(numPoints=0, classBand='ref', region=AREAS[area], scale=30,
                                     classValues=[1], classPoints=[600], seed=SEED, tileScale=4, geometries=False)
        d = pd.DataFrame([f['properties'] for f in pts.getInfo()['features']]).dropna()
        d['canopy bin'] = pd.cut(d['canopy2000'], [-1, 0, 19, 59, 100], labels=['0%', '1-19%', '20-59%', '60-100%'])
        for cls, name in CLASSES.items():
            rows.append({'Area': area, 'Map year': year, 'Group': 'Our final map says', 'Value': name,
                         'Share of WorldCover tree points (%)': round(100 * (d['our'] == cls).mean(), 1)})
        for b in ['0%', '1-19%', '20-59%', '60-100%']:
            rows.append({'Area': area, 'Map year': year, 'Group': 'Hansen canopy 2000', 'Value': b,
                         'Share of WorldCover tree points (%)': round(100 * (d['canopy bin'] == b).mean(), 1)})
diag_forest = pd.DataFrame(rows).pivot_table(index=['Group', 'Value'], columns=['Area', 'Map year'],
                                             values='Share of WorldCover tree points (%)', sort=False)
diag_forest


In [ ]:
# Forest area per epoch: final maps (Hansen canopy rule) next to the Random Forest forest class
rows = []
for area in AREAS:
    for year in YEARS:
        rf_a = class_areas(CLASSIFIED_RF[(area, year)], AREAS[area])
        rows.append({'Area': area, 'Year': year,
                     'Forest, final map (km2)': round(AREA_CACHE[(area, year)].get(4, 0.0), 2),
                     'Forest, Random Forest (km2)': round(rf_a.get(4, 0.0), 2)})
forest_compare = pd.DataFrame(rows)
forest_compare


## 8. Save results  —  **RUN THIS CELL BEFORE THE FIGURES**
Writes `ttd_lulc_results.xlsx` (accuracy, pooled accuracy, scenes, sample counts, split, areas, GHSL/JRC comparison, forest checks, transitions) and downloads it.

In [ ]:
short = {'Tirumala': 'Tml', 'Tirupati': 'Tpt'}     # short sheet names (Excel limit 31 characters)
with pd.ExcelWriter('ttd_lulc_results.xlsx') as xw:
    acc_df.to_excel(xw, sheet_name='Accuracy', index=False)
    pooled_df.to_excel(xw, sheet_name='Accuracy_pooled', index=False)
    ndvi_by_class.to_excel(xw, sheet_name='NDVI_by_class')
    scene_df.to_excel(xw, sheet_name='Scenes', index=False)
    sample_counts.to_excel(xw, sheet_name='Sample_counts')
    pd.DataFrame(SPLIT_INFO).to_excel(xw, sheet_name='Split', index=False)
    ghsl_jrc.to_excel(xw, sheet_name='GHSL_JRC_comparison', index=False)
    if 'forest_check' in globals():
        forest_check.to_excel(xw, sheet_name='Forest_check', index=False)
    if 'diag_forest' in globals():
        diag_forest.to_excel(xw, sheet_name='Forest_diagnostic')
    if 'forest_compare' in globals():
        forest_compare.to_excel(xw, sheet_name='Forest_compare', index=False)
    for area, t in area_tables.items():
        t.to_excel(xw, sheet_name=f'Area_{area}')
    for (area, y1, y2), m in TRANSITIONS.items():
        m.to_excel(xw, sheet_name=f'{short[area]}_{y1}_{y2}')
print('Saved ttd_lulc_results.xlsx')

try:
    from google.colab import files
    files.download('ttd_lulc_results.xlsx')
except Exception:
    print('Download the file from the Colab file panel (folder icon, left side).')

## 9. Figures (300 dpi, no captions inside the images; put captions in the chapter text)
Fig 1 study area (Landsat 2025 true colour), Fig 2 and Fig 3 maps per year, Fig 4 area by class, Fig 5 net change, Fig 6 transitions.
All six are saved as PNG and downloaded together as `TTD_figures.zip`.

Note: Fig 5 (net change) is produced here but is not used in the chapter; the chapter's Fig 5 is the transition matrix figure (file `Fig6_transitions.png`).

In [ ]:
import requests, io, glob
from PIL import Image
from matplotlib.patches import Patch, Rectangle

BOUNDS = BOX
OVERVIEW = [79.25, 13.50, 79.65, 13.78]
CLASS_COL = {c: '#' + PALETTE[c - 1] for c in CLASS_IDS}
FS = 7   # base font size


def thumb(image, bounds, dims=900):
    """Download a PNG of an Earth Engine image for the given [W,S,E,N] box."""
    region = ee.Geometry.Rectangle(bounds)
    url = image.getThumbURL({'region': region, 'dimensions': dims, 'format': 'png', 'crs': 'EPSG:4326'})
    r = requests.get(url, timeout=180); r.raise_for_status()
    return np.array(Image.open(io.BytesIO(r.content)).convert('RGBA'))


def decorate(ax, b, scale_km=5, col='k'):
    W, S, E, N = b
    ax.set_aspect(1 / np.cos(np.radians((S + N) / 2)))
    ax.tick_params(labelsize=FS - 1)
    dlon = scale_km / (111.32 * np.cos(np.radians((S + N) / 2)))
    x0, y0 = W + 0.05 * (E - W), S + 0.06 * (N - S)
    ax.plot([x0, x0 + dlon], [y0, y0], color=col, lw=2, solid_capstyle='butt')
    ax.text(x0 + dlon / 2, y0 + 0.035 * (N - S), f'{scale_km} km', ha='center', fontsize=FS - 1, color=col)
    ax.annotate('', xy=(0.93, 0.90), xytext=(0.93, 0.77), xycoords='axes fraction',
                arrowprops=dict(arrowstyle='-|>', color=col, lw=1.2))
    ax.text(0.93, 0.915, 'N', transform=ax.transAxes, ha='center', va='bottom', fontsize=FS, color=col)


# ---- Figure 1: study area on a Landsat 2025 true-colour image ----
ov_img, _ = get_composite(2025, ee.Geometry.Rectangle(OVERVIEW))
rgb = thumb(ov_img.visualize(bands=['red', 'green', 'blue'], min=0.02, max=0.28, gamma=1.2), OVERVIEW)
fig, ax = plt.subplots(figsize=(5.2, 4.2), dpi=300)
ax.imshow(rgb, extent=[OVERVIEW[0], OVERVIEW[2], OVERVIEW[1], OVERVIEW[3]])
for name, col in [('Tirumala', 'yellow'), ('Tirupati', 'cyan')]:
    W, S, E, N = BOUNDS[name]
    ax.add_patch(Rectangle((W, S), E - W, N - S, fill=False, edgecolor=col, lw=1.4))
    ax.text(W + 0.003, N - 0.012, name, color=col, fontsize=FS + 1, weight='bold')
decorate(ax, OVERVIEW, 5, 'white')
ax.set_xlabel('Longitude (°E)', fontsize=FS); ax.set_ylabel('Latitude (°N)', fontsize=FS)
plt.tight_layout(); plt.savefig('Fig1_study_area.png', dpi=300); plt.show()

# ---- Figures 2 and 3: LULC maps for every year, one figure per area ----
legend_items = [Patch(facecolor=CLASS_COL[c], edgecolor='k', label=CLASSES[c]) for c in CLASS_IDS]
for k, area in enumerate(AREAS, start=2):
    b = BOUNDS[area]
    fig, axes = plt.subplots(2, 3, figsize=(6.85, 5.2), dpi=300)
    for ax, year in zip(axes.flat, YEARS):
        img = CLASSIFIED[(area, year)].visualize(min=1, max=5, palette=PALETTE)
        ax.imshow(thumb(img, b), extent=[b[0], b[2], b[1], b[3]])
        ax.set_title(str(year), fontsize=FS + 1)
        decorate(ax, b, 2 if area == 'Tirumala' else 5)
        ax.set_xticks([]); ax.set_yticks([])
    fig.legend(handles=legend_items, loc='lower center', ncol=len(legend_items), fontsize=FS, frameon=False)
    fig.suptitle(f'Land use/land cover, {area}', fontsize=FS + 2)
    plt.tight_layout(rect=[0, 0.05, 1, 0.97])
    fig.subplots_adjust(hspace=0.25)            # stops the year titles overlapping the row above
    plt.savefig(f'Fig{k}_LULC_{area}.png', dpi=300); plt.show()

# ---- Figure 4: area by class over time (built-up and water on top, forest below) ----
cols4 = [CLASS_COL[c] for c in CLASS_IDS if c != 5]          # colours for Figure 5
fig, axes = plt.subplots(2, 2, figsize=(6.85, 4.6), dpi=300, sharex=True)
for j, area in enumerate(AREAS):
    t = area_tables[area]
    for name, cid in [('Built-up', 1), ('Water bodies', 3)]:
        axes[0, j].plot(YEARS, t.loc[name, YEARS].values, marker='o', ms=3, lw=1.2,
                        color=CLASS_COL[cid], label=name)
    axes[1, j].plot(YEARS, t.loc['Forest cover', YEARS].values, marker='o', ms=3, lw=1.2,
                    color=CLASS_COL[4], label='Forest cover')
    axes[0, j].set_title(area, fontsize=FS + 1)
    axes[0, j].set_ylim(bottom=0)
    axes[1, j].set_ylim(0, t.loc['Forest cover', YEARS].max() * 1.15)
    for i in (0, 1):
        axes[i, j].set_ylabel('Area (km$^2$)', fontsize=FS)
        axes[i, j].tick_params(labelsize=FS)
        axes[i, j].grid(alpha=0.3)
        axes[i, j].legend(fontsize=FS - 1, frameon=False)
    axes[1, j].set_xlabel('Year', fontsize=FS)
plt.tight_layout(); plt.savefig('Fig4_area_trends.png', dpi=300); plt.show()

# ---- Figure 5: net change 2000-2025 ----
fig, axes = plt.subplots(1, 2, figsize=(6.85, 3), dpi=300)
for ax, area in zip(axes, AREAS):
    vals = area_tables[area].loc[four, 'Change 2000-2025 (km2)']
    ax.bar(four, vals.values, color=cols4, edgecolor='k', lw=0.5)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_title(area, fontsize=FS + 1); ax.set_ylabel('Net change 2000-2025 (km$^2$)', fontsize=FS)
    ax.tick_params(labelsize=FS - 1); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig('Fig5_net_change.png', dpi=300); plt.show()

# ---- Figure 6: transition matrices 2000 -> 2025 ----
fig, axes = plt.subplots(1, 2, figsize=(6.85, 3.4), dpi=300)
for ax, area in zip(axes, AREAS):
    m = TRANSITIONS[(area, 2000, 2025)]
    ax.imshow(m.values, cmap='Blues')
    ax.set_xticks(range(m.shape[1])); ax.set_xticklabels(m.columns, rotation=35, ha='right', fontsize=FS - 1)
    ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index, fontsize=FS - 1)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]
            ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=FS - 1,
                    color='white' if v > m.values.max() / 2 else 'black')
    ax.set_title(area, fontsize=FS + 1)
    ax.set_xlabel('Class in 2025', fontsize=FS); ax.set_ylabel('Class in 2000', fontsize=FS)
plt.tight_layout(); plt.savefig('Fig6_transitions.png', dpi=300); plt.show()

# ---- zip and download all figures in one file ----
import zipfile
with zipfile.ZipFile('TTD_figures.zip', 'w') as z:
    for f in sorted(glob.glob('Fig*.png')):
        z.write(f)
print('Figures saved:', sorted(glob.glob('Fig*.png')))
try:
    from google.colab import files
    files.download('TTD_figures.zip')
except Exception:
    print('Download TTD_figures.zip from the Colab file panel (folder icon, left side).')


## 10. Optional: interactive map and export (skip if the Earth Engine quota is exhausted)

In [ ]:
view_area = 'Tirupati'
vis = {'min': 1, 'max': 5, 'palette': PALETTE}
M = geemap.Map(basemap='HYBRID')
M.centerObject(AREAS[view_area], 11)
for year in YEARS:
    M.addLayer(CLASSIFIED[(view_area, year)], vis, f'LULC {year}', shown=(year == 2025))
M.add_legend(title='Land cover', legend_dict={CLASSES[c]: '#' + PALETTE[c - 1] for c in CLASS_IDS})
M

In [ ]:
# Export the classified maps to Google Drive (folder TTD_LULC) for the final map layouts in QGIS
from google.colab import drive
drive.mount('/content/drive')
for (area, year), img in CLASSIFIED.items():
    ee.batch.Export.image.toDrive(image=img, description=f'LULC_{area}_{year}', folder='TTD_LULC',
                                  region=AREAS[area], scale=30, maxPixels=1e10).start()
print('Export tasks started - check the Tasks tab at code.earthengine.google.com or your Drive folder.')

### Notes for the Methods and Limitations sections (fill numbers only from the outputs above)
* Landsat Collection 2 Level 2 surface reflectance, 30 m, cloud masked with `QA_PIXEL`; Jan–Apr median composites from three years per map (2000: 2000–2002; others: year−1 to year+1).
* Landsat 5/7 converted to Landsat 8 level with the ETM+→OLI OLS coefficients of Roy et al. (2016); applied to Landsat 5 by assumption.
* Training and validation points from GHSL GHS-BUILT-S R2023A (built-up), JRC Global Surface Water (water), Hansen Global Forest Change v1.13 (forest); accuracy measures agreement with these datasets, not field truth.
* One Random Forest (200 trees, 9 bands) trained on points pooled from all years and both areas; 3x3 majority filter; train/validation split by ~1 km spatial blocks.
* Final maps: built-up and water from the Random Forest; forest = Hansen canopy ≥ 20% in 2000 minus loss; other land = the rest. Accuracy figures describe the Random Forest maps. The final forest layer is checked only against ESA WorldCover (Section 7b; different tree-cover definition, so low Kappa is expected and is reported as such); other land is not assessed.
* Roads were not classified (no free historical road data; roads narrower than 30 m).
* Hansen annual loss to 2025 is used; its forest gain layer covers only 2000–2012 and is not used, so forest gain and canopy thinning are not measured. The GHSL 2025 epoch may be a projection (check Pesaresi and Politis 2023); the 2025 water reference uses 2021.
* Kappa limit 0.75 was fixed after the first results were seen. Tirupati built-up in 2015 is lower than the GHSL footprint suggests: report net 2000–2025 change and the GHSL comparison table.
* "Other land" is a background class.